In [12]:
import polars as pl

# 1. Daten laden (wie im vorherigen Skript)
df = pl.read_csv(r"..\..\..\Raw_data_sets\kiel_specific_stations_raw.csv")

# 2. Zeitstempel-Logik anwenden (damit wir echte Zeit-Duplikate finden)
df = df.with_columns([
    pl.col("Datum").cast(pl.String).str.to_date("%y%m%d"),
    pl.col("Stunde").cast(pl.Int8)
]).with_columns(
    pl.datetime(
        pl.col("Datum").dt.year(),
        pl.col("Datum").dt.month(),
        pl.col("Datum").dt.day(),
        pl.col("Stunde") - 1
    ).alias("timestamp")
)

# 3. Duplikate finden
duplicates = (
    df.group_by(["timestamp", "Zst"])
    .agg(pl.len().alias("count"), pl.all().exclude(["timestamp", "Zst"]))
    .filter(pl.col("count") > 1)
    .sort("timestamp")
)

print(f"Anzahl der Kombinationen mit Duplikaten: {duplicates.height}")
# Zeige die ersten paar Duplikate an
print(duplicates.head())

Anzahl der Kombinationen mit Duplikaten: 0
shape: (0, 62)
┌─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬───┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┐
│ tim ┆ Zst ┆ cou ┆ TKN ┆ Lan ┆ Str ┆ Str ┆ Dat ┆ Wot ┆ Fah ┆ Stu ┆ KFZ ┆ K_K ┆ … ┆ K_B ┆ LoA ┆ K_L ┆ Lzg ┆ K_L ┆ Sat ┆ K_S ┆ Son ┆ K_S ┆ yea ┆ mon ┆ var │
│ est ┆ --- ┆ nt  ┆ R   ┆ d   ┆ kla ┆ num ┆ um  ┆ ag  ┆ rtz ┆ nde ┆ _R1 ┆ FZ_ ┆   ┆ us_ ┆ _R2 ┆ oA_ ┆ _R2 ┆ zg_ ┆ _R2 ┆ at_ ┆ _R2 ┆ on_ ┆ r   ┆ th  ┆ ian │
│ amp ┆ i64 ┆ --- ┆ --- ┆ --- ┆ s   ┆ --- ┆ --- ┆ --- ┆ w   ┆ --- ┆ --- ┆ R1  ┆   ┆ R2  ┆ --- ┆ R2  ┆ --- ┆ R2  ┆ --- ┆ R2  ┆ --- ┆ R2  ┆ --- ┆ --- ┆ t   │
│ --- ┆     ┆ u32 ┆ lis ┆ lis ┆ --- ┆ lis ┆ lis ┆ lis ┆ --- ┆ lis ┆ lis ┆ --- ┆   ┆ --- ┆ lis ┆ --- ┆ lis ┆ --- ┆ lis ┆ --- ┆ lis ┆ --- ┆ lis ┆ lis ┆ --- │
│ dat ┆     ┆     ┆ t[i ┆ t[i ┆ lis ┆ t[s ┆ t[d ┆ t[s ┆ lis ┆ t[i ┆ t[s ┆ lis ┆   ┆ lis ┆ t[s ┆ lis ┆ t[s ┆ lis ┆ t[s ┆ lis ┆ t[s ┆ lis ┆ t[i ┆ t[i ┆ lis │
│ eti 

In [2]:
import polars as pl
import os

# 1. PATH CONFIGURATION
# Navigating from: DataScienceProject\Data-Science-Project---Group-16\Code\CSV-Transformation
# To: DataScienceProject\Raw_data_sets
DATA_ROOT = os.path.join("..", "..", "..", "Raw_data_sets")
TRAFFIC_FILE = os.path.join(DATA_ROOT, "kiel_specific_stations_raw.csv")
WEATHER_FILE = os.path.join(DATA_ROOT, "weather_hourly.csv")
OUTPUT_FILE = "master_wide_hourly.parquet"

def process_traffic():
    print(f"Loading traffic data: {TRAFFIC_FILE}")
    
    # Use scan_csv for efficiency
    q = (
        pl.scan_csv(TRAFFIC_FILE, separator=",", infer_schema_length=10000)
        .with_columns([
            # Convert '210101' -> Date
            pl.col("Datum").cast(pl.String).str.to_date("%y%m%d"),
            pl.col("Stunde").cast(pl.Int8)
        ])
        .with_columns(
            # Generate timestamp: hour 1 = 00:00
            pl.datetime(
                pl.col("Datum").dt.year(),
                pl.col("Datum").dt.month(),
                pl.col("Datum").dt.day(),
                pl.col("Stunde") - 1
            ).alias("timestamp")
        )
    )
    
    # Collecting to perform the Pivot
    df_traffic = q.collect()
    
    # Automatically identify all measurement columns (excluding metadata)
    metadata_cols = ["TKNR", "Zst", "Land", "Strklas", "Strnum", "Datum", 
                     "Wotag", "Fahrtzw", "Stunde", "year", "month", "variant", "timestamp"]
    measurement_cols = [c for c in df_traffic.columns if c not in metadata_cols]
    
    print(f"Pivoting {len(measurement_cols)} measurements for all stations...")
    
    # Pivot: One row per timestamp, columns split by station (TKNR)
    df_wide = df_traffic.pivot(
        values=measurement_cols,
        index="timestamp",
        on="TKNR"
    )
    
    return df_wide.lazy()

def process_weather():
    print(f"Loading weather data: {WEATHER_FILE}")
    return (
        pl.scan_csv(WEATHER_FILE)
        .with_columns(
            # Format: 01.01.2021 00:00
            pl.col("date").str.to_datetime("%d.%m.%Y %H:%M").alias("timestamp")
        )
        .drop("date")
        .unique(subset=["timestamp"])
    )

# --- EXECUTION ---

try:
    traffic_lazy = process_traffic()
    weather_lazy = process_weather()

    print("Joining datasets...")
    final_df = (
        traffic_lazy.join(weather_lazy, on="timestamp", how="left")
        .sort("timestamp")
        .collect()
    )

    # Save as Parquet for high-speed loading in notebooks
    final_df.write_parquet(OUTPUT_FILE)

    print("-" * 30)
    print("SUCCESS")
    print(f"Master file created: {OUTPUT_FILE}")
    print(f"Dimensions: {final_df.height} hours x {final_df.width} columns")

except Exception as e:
    print(f"An error occurred: {e}")
    print("Check if the file names and paths are exactly correct.")

Loading traffic data: ..\..\..\Raw_data_sets\kiel_stations_raw.csv
Pivoting 48 measurements for all stations...
An error occurred: found multiple elements in the same group, please specify an aggregation function
Check if the file names and paths are exactly correct.


In [ ]:
import polars as pl
from datetime import datetime

start = datetime(2021, 1, 1, 0, 0)
end = datetime(2025, 12, 31, 23, 0)


# Initiating master data frame

master_df = (
    pl.datetime_range(start, end, interval="1h", eager=True)
    .alias("timestamp")
    .to_frame()
)

In [ ]:
# getting the data

df_traffic = 

In [8]:
import openmeteo_requests
import polars as pl
import requests_cache
from retry_requests import retry
from datetime import datetime, timedelta

# 1. API Client Setup
cache_session = requests_cache.CachedSession('.cache', expire_after=-1)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

# 2. Parameter (Wetterdaten für Anfang 2026)
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": [54.3213, 54.36464],
    "longitude": [10.1349, 10.12133],
    "start_date": "2026-01-01",
    "end_date": "2026-01-02",
    "hourly": ["temperature_2m", "relative_humidity_2m", "precipitation", "snowfall", "weather_code", "wind_speed_10m", "wind_gusts_10m"],
}

# 3. API Abruf
responses = openmeteo.weather_api(url, params=params)

# Liste für die DataFrames
all_dfs = []

# 4. Daten verarbeiten
for i, response in enumerate(responses):
    hourly = response.Hourly()
    start = datetime.fromtimestamp(hourly.Time())
    end = datetime.fromtimestamp(hourly.TimeEnd())
    interval = timedelta(seconds=hourly.Interval())
    
    df_loc = pl.DataFrame({
        "time": pl.datetime_range(start, end, interval, eager=True, closed="left"),
        "temp": hourly.Variables(0).ValuesAsNumpy(),
        "hum": hourly.Variables(1).ValuesAsNumpy(),
        "prec": hourly.Variables(2).ValuesAsNumpy(),
        "snow": hourly.Variables(3).ValuesAsNumpy(),
        "code": hourly.Variables(4).ValuesAsNumpy(),
        "wind": hourly.Variables(5).ValuesAsNumpy(),
        "gust": hourly.Variables(6).ValuesAsNumpy(),
        "location": f"L{i+1}" # Kurze ID für Spaltennamen
    })
    all_dfs.append(df_loc)

# 5. Alle Daten untereinander hängen (Long Format)
df_long = pl.concat(all_dfs)

# 6. Transformation: Vergleichstabelle (Alle Werte nebeneinander)
# Wir nutzen pivot, um für jede Location eigene Spalten zu bekommen
comparison_table = df_long.pivot(
    index="time",
    on="location",
    values=["temp", "hum", "prec", "snow", "code", "wind", "gust"]
).sort("time")

# Spalten sortieren für bessere Lesbarkeit (Time, dann L1-Werte, dann L2-Werte)
# Oder alternativ: Variablenweise (Temp_L1, Temp_L2, Hum_L1, Hum_L2...)
ordered_columns = ["time"] + sorted([c for c in comparison_table.columns if c != "time"])
comparison_table = comparison_table.select(ordered_columns)

# 7. Ausgabe
print("--- VOLLSTÄNDIGE VERGLEICHSTABELLE ---")
print(comparison_table)

# Optional: Speichern als CSV für Excel
comparison_table.write_csv("wetter_vergleich.csv")

--- VOLLSTÄNDIGE VERGLEICHSTABELLE ---
shape: (48, 15)
┌──────────────┬─────────┬─────────┬───────────┬───┬─────────┬─────────┬───────────┬───────────┐
│ time         ┆ code_L1 ┆ code_L2 ┆ gust_L1   ┆ … ┆ temp_L1 ┆ temp_L2 ┆ wind_L1   ┆ wind_L2   │
│ ---          ┆ ---     ┆ ---     ┆ ---       ┆   ┆ ---     ┆ ---     ┆ ---       ┆ ---       │
│ datetime[μs] ┆ f32     ┆ f32     ┆ f32       ┆   ┆ f32     ┆ f32     ┆ f32       ┆ f32       │
╞══════════════╪═════════╪═════════╪═══════════╪═══╪═════════╪═════════╪═══════════╪═══════════╡
│ 2026-01-01   ┆ 51.0    ┆ 51.0    ┆ 39.959999 ┆ … ┆ 4.7     ┆ 4.85    ┆ 21.788923 ┆ 22.963936 │
│ 01:00:00     ┆         ┆         ┆           ┆   ┆         ┆         ┆           ┆           │
│ 2026-01-01   ┆ 51.0    ┆ 51.0    ┆ 42.119999 ┆ … ┆ 4.15    ┆ 4.45    ┆ 22.459644 ┆ 23.031561 │
│ 02:00:00     ┆         ┆         ┆           ┆   ┆         ┆         ┆           ┆           │
│ 2026-01-01   ┆ 53.0    ┆ 51.0    ┆ 49.68     ┆ … ┆ 3.5     ┆ 3.95    ┆

In [10]:
import openmeteo_requests
import polars as pl
import requests_cache
from retry_requests import retry
from datetime import datetime, timedelta

# 1. API Client Setup
cache_session = requests_cache.CachedSession('.cache', expire_after=-1)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

# 2. Parameter für 3 Standorte
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": [54.3213, 54.36464, 54.31122],
    "longitude": [10.1349, 10.12133, 10.07769],
    "start_date": "2026-01-01",
    "end_date": "2026-01-02",
    "hourly": ["temperature_2m", "relative_humidity_2m", "precipitation", "snowfall", "weather_code", "wind_speed_10m", "wind_gusts_10m"],
}

# 3. Daten abrufen
responses = openmeteo.weather_api(url, params=params)

all_dfs = []

# 4. Daten in Polars DataFrames umwandeln
for i, response in enumerate(responses):
    hourly = response.Hourly()
    start = datetime.fromtimestamp(hourly.Time())
    end = datetime.fromtimestamp(hourly.TimeEnd())
    interval = timedelta(seconds=hourly.Interval())
    
    df_loc = pl.DataFrame({
        "time": pl.datetime_range(start, end, interval, eager=True, closed="left"),
        "temp": hourly.Variables(0).ValuesAsNumpy(),
        "hum": hourly.Variables(1).ValuesAsNumpy(),
        "prec": hourly.Variables(2).ValuesAsNumpy(),
        "snow": hourly.Variables(3).ValuesAsNumpy(),
        "code": hourly.Variables(4).ValuesAsNumpy(),
        "wind": hourly.Variables(5).ValuesAsNumpy(),
        "gust": hourly.Variables(6).ValuesAsNumpy(),
        "loc": f"L{i+1}"  # Kennung L1, L2, L3
    })
    all_dfs.append(df_loc)

# 5. Zusammenführen und Pivotieren für den Nebeneinander-Vergleich
df_all = pl.concat(all_dfs)

# Erstellt eine Tabelle, in der pro Zeitstempel alle Werte aller Orte stehen
comparison_table = df_all.pivot(
    index="time",
    on="loc",
    values=["temp", "hum", "prec", "snow", "code", "wind", "gust"]
).sort("time")

# 6. Spalten sortieren, damit gleiche Variablen beieinander stehen (z.B. temp_L1, temp_L2, temp_L3)
cols = comparison_table.columns
time_col = ["time"]
variable_cols = sorted([c for c in cols if c != "time"])
comparison_table = comparison_table.select(time_col + variable_cols)

# --- Ausgabe ---
# Erhöht das Limit der angezeigten Spalten für das Terminal
pl.Config.set_tbl_cols(25) 

print("--- VERGLEICHSTABELLE (3 STANDORTE) ---")
print(comparison_table)

# Beispiel: Nur Temperaturen vergleichen
print("\n--- NUR TEMPERATUREN ---")
print(comparison_table.select(pl.col("^time|temp_.*$")))
comparison_table.write_csv("wetter_vergleich.csv")

--- VERGLEICHSTABELLE (3 STANDORTE) ---
shape: (48, 22)
┌─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┐
│ tim ┆ cod ┆ cod ┆ cod ┆ gus ┆ gus ┆ gus ┆ hum ┆ hum ┆ hum ┆ pre ┆ pre ┆ pre ┆ sno ┆ sno ┆ sno ┆ tem ┆ tem ┆ tem ┆ win ┆ win ┆ win │
│ e   ┆ e_L ┆ e_L ┆ e_L ┆ t_L ┆ t_L ┆ t_L ┆ _L1 ┆ _L2 ┆ _L3 ┆ c_L ┆ c_L ┆ c_L ┆ w_L ┆ w_L ┆ w_L ┆ p_L ┆ p_L ┆ p_L ┆ d_L ┆ d_L ┆ d_L │
│ --- ┆ 1   ┆ 2   ┆ 3   ┆ 1   ┆ 2   ┆ 3   ┆ --- ┆ --- ┆ --- ┆ 1   ┆ 2   ┆ 3   ┆ 1   ┆ 2   ┆ 3   ┆ 1   ┆ 2   ┆ 3   ┆ 1   ┆ 2   ┆ 3   │
│ dat ┆ --- ┆ --- ┆ --- ┆ --- ┆ --- ┆ --- ┆ f32 ┆ f32 ┆ f32 ┆ --- ┆ --- ┆ --- ┆ --- ┆ --- ┆ --- ┆ --- ┆ --- ┆ --- ┆ --- ┆ --- ┆ --- │
│ eti ┆ f32 ┆ f32 ┆ f32 ┆ f32 ┆ f32 ┆ f32 ┆     ┆     ┆     ┆ f32 ┆ f32 ┆ f32 ┆ f32 ┆ f32 ┆ f32 ┆ f32 ┆ f32 ┆ f32 ┆ f32 ┆ f32 ┆ f32 │
│ me[ ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     │
│ μs] 

In [1]:
import polars as pl

# 1. Verbindung zur Datei herstellen (Lazy-Modus für große Dateien)
df = pl.scan_csv("../../kiel_specific_stations_raw.csv")

# 2. Nur die relevanten Spalten auswählen und Duplikate entfernen
# (Ersetze 'station_name', 'lat', 'lon' durch deine echten Spaltennamen)
unique_stations = (
    df.select([
        "Zst",
        "Strklas",
        "Strnum",
        "name",
        "lat", 
        "lon",
        "richtung_1",
        "richtung_2"
    ])
    .unique()
    .collect() # Erst hier wird die Berechnung tatsächlich ausgeführt
)

# 3. Ergebnis speichern
unique_stations.write_csv("meta_data_used_stations.csv")

print(unique_stations)

shape: (8, 8)
┌──────┬─────────┬────────┬──────────────────────┬───────────┬───────────┬────────────┬────────────┐
│ Zst  ┆ Strklas ┆ Strnum ┆ name                 ┆ lat       ┆ lon       ┆ richtung_1 ┆ richtung_2 │
│ ---  ┆ ---     ┆ ---    ┆ ---                  ┆ ---       ┆ ---       ┆ ---        ┆ ---        │
│ i64  ┆ str     ┆ str    ┆ str                  ┆ f64       ┆ f64       ┆ str        ┆ str        │
╞══════╪═════════╪════════╪══════════════════════╪═══════════╪═══════════╪════════════╪════════════╡
│ 1116 ┆ B       ┆   76   ┆ Gettorf ( Wulfshagen ┆ 54.392371 ┆ 10.020723 ┆ O          ┆ W          │
│      ┆         ┆        ┆ )                    ┆           ┆           ┆            ┆            │
│ 1194 ┆ A       ┆  215   ┆ Kiel-West            ┆ 54.311138 ┆ 10.075224 ┆ N          ┆ S          │
│ 1112 ┆ B       ┆  503   ┆ Kiel-Holtenau 2      ┆ 54.361172 ┆ 10.121769 ┆ N          ┆ S          │
│ 1111 ┆ B       ┆  503   ┆ Kiel-Holtenau I      ┆ 54.368315 ┆ 10.121564 ┆ N 